<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/SAHI_PBCInlet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing all the necessary packages

In [1]:
import sys
if 'google.colab' in sys.modules:
    %pip install sahi ultralytics

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image
from ultralytics import YOLO
from PIL import Image as PILImage, ImageDraw, ImageFont
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.12.0.88
    Uninstalling opencv-python-4.12.0.88:
      Successfully uninstalled opencv-python-4.12.0.88
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Import Modules and Download Resources

In [2]:
model_path = "yolov8n.pt"
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.3,
    device="cuda:0"
)

I am running a pre-trained model, yolov8n in this case to see how well it performs on the Jupiter Inlet imagery, where the boats (objects) are only 5-10 pixels in length.  

In [10]:
# Install dependencies (Colab only)
import sys
if "google.colab" in sys.modules:
    !pip install -q ultralytics pillow

import os
import json
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

# Define image folder explicitly
image_folder = "/content/Jupiter_Inlet"

# Load ground truth to get filename to image ID mapping for consistent evaluation
ground_truth_json = "/content/Jupiter_Inlet/ground_truth_predicitions.coco.json"
with open(ground_truth_json) as f:
    gt_data = json.load(f)
filename_to_id = {img['file_name']: img['id'] for img in gt_data['images']}

model = YOLO("yolov8n.pt")  # or your trained model

yolo_predictions = {
    "images": [],
    "annotations": [],
    "categories": gt_data["categories"] # Use categories from ground truth for consistency
}

annotation_id = 1

# Iterate through filenames and use the consistent image_id from the ground truth mapping
for filename in sorted(os.listdir(image_folder)):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    current_image_id = filename_to_id.get(filename)
    if current_image_id is None:
        print(f"Warning: Image {filename} not found in ground truth. Skipping.")
        continue

    image_path = os.path.join(image_folder, filename)
    results = model.predict(image_path, conf=0.3)
    boxes = results[0].boxes  # XYXY format

    # Save image metadata
    image = Image.open(image_path)
    yolo_predictions["images"].append({
        "id": current_image_id,
        "file_name": filename,
        "width": image.width,
        "height": image.height
    })

    for box in boxes:
        x_min, y_min, x_max, y_max = box.xyxy[0]
        score = box.conf[0].item()
        # The normalize_coco function will standardize category IDs to 1 (BOAT_CAT_ID).

        category_id = 1

        yolo_predictions["annotations"].append({
            "id": annotation_id,
            "image_id": current_image_id,
            "category_id": category_id,
            "bbox": [float(x_min), float(y_min), float(x_max - x_min), float(y_max - y_min)],
            "score": float(score)
        })
        annotation_id += 1


image 1/1 /content/Jupiter_Inlet/s241527w_jpg.rf.RzbGR3Qs98ylL67za1Eo.jpg: 480x640 (no detections), 158.0ms
Speed: 4.0ms preprocess, 158.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/Jupiter_Inlet/s241727k_jpg.rf.iScCKzvmzbLyO2iGTham.jpg: 480x640 (no detections), 155.2ms
Speed: 3.7ms preprocess, 155.2ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/Jupiter_Inlet/s251125h_jpg.rf.FJ2GTEjqflRczR3ENlBJ.jpg: 480x640 1 car, 153.7ms
Speed: 3.7ms preprocess, 153.7ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/Jupiter_Inlet/s251223c_jpg.rf.nSqFwsDpd0B3fIiwITV6.jpg: 480x640 (no detections), 161.6ms
Speed: 3.8ms preprocess, 161.6ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/Jupiter_Inlet/s251506k_jpg.rf.M7NKqN8XDaABc22gL7Ze.jpg: 480x640 (no detections), 165.8ms
Speed: 3.7ms preprocess, 165.8ms inference, 0.8ms postprocess per image a

Running SAHI

In [12]:
#loading the required packages
import os
import json
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image
from sahi import AutoDetectionModel

#setting the place of my folder
image_folder = "/content/Jupiter_Inlet"
output_folder = "/content/Jupiter_Inlet/output_sahi"
os.makedirs(output_folder, exist_ok=True)

ground_truth_json = "/content/Jupiter_Inlet/ground_truth_predicitions.coco.json"
with open(ground_truth_json) as f:
    gt = json.load(f)
filename_to_id = {img['file_name']: img['id'] for img in gt['images']}

#loading my model and setting the threshold
model_path = "yolov8n.pt"
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.3,
    device="cuda:0"
)


#looping over all my images with many different SAHI window sizes

slice_sizes = [150]

for slice_size in slice_sizes:
    print(f"\n=== Running SAHI with slice size {slice_size}x{slice_size} ===")

    predictions = []

    for filename in os.listdir(image_folder):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(image_folder, filename)
        print(f"Processing {filename}...")

        result = get_sliced_prediction(
            image=image_path,
            detection_model=detection_model,
            slice_height=slice_size,
            slice_width=slice_size,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
        )

        # Only boats
        boat_preds = [
            obj for obj in result.object_prediction_list
            if obj.category.name.lower() == "boat"
        ]

        for obj in boat_preds:
            bbox = obj.bbox.to_xywh()
            predictions.append({
                "image_id": filename_to_id[filename],
                "category_id": 1,  # boat
                "bbox": [float(b) for b in bbox],
                "score": float(obj.score.value),
            })

    # Save per-window-size JSON
    pred_json_path = os.path.join(
        output_folder, f"sahi_predictions_slice_{slice_size}.json"
    )

    with open(pred_json_path, "w") as f:
        json.dump(predictions, f, indent=4)

    print(f"Saved {len(predictions)} predictions to {pred_json_path}")


=== Running SAHI with slice size 150x150 ===
Processing s261205u_jpg.rf.7zQR37dlY7zpUApTuvRt.jpg...
Performing prediction on 999 slices.
Processing s281608g_jpg.rf.zJ9LzqMMCA9vgtD0cCtC.jpg...
Performing prediction on 999 slices.
Processing s251809g_jpg.rf.4vDqdcBH620kG6JMvGry.jpg...
Performing prediction on 999 slices.
Processing s271723a_jpg.rf.m7R6C7gvH6IlXxmTOnqh.jpg...
Performing prediction on 999 slices.
Processing s261605t_jpg.rf.gz3HIbQzVsLZDVByYDWb.jpg...
Performing prediction on 999 slices.
Processing s261320w_jpg.rf.14FfIFe6dR8NMZh1KtJB.jpg...
Performing prediction on 999 slices.
Processing s241727k_jpg.rf.iScCKzvmzbLyO2iGTham.jpg...
Performing prediction on 999 slices.
Processing s281105l_jpg.rf.L0RLflk7eDzMY8WwklO4.jpg...
Performing prediction on 999 slices.
Processing s270702m_jpg.rf.w8pXrW3QfOtvJ6JilECa.jpg...
Performing prediction on 999 slices.
Processing s271127r_jpg.rf.zFiejHNu4WtrQSZkGEgi.jpg...
Performing prediction on 999 slices.
Processing s291102v_jpg.rf.lZkpHfL

Evaluation

In [14]:
#lo0ading the necessary packages
import json
from pycocotools.coco import COCO
from sklearn.metrics import confusion_matrix
import os

# =========================
# PATHS
# =========================
GT_JSON = "/content/Jupiter_Inlet/ground_truth_predicitions.coco.json"

SAHI_FILES = {
    150: "/content/Jupiter_Inlet/output_sahi/sahi_predictions_slice_150.json",
    250: "/content/Jupiter_Inlet/output_sahi/sahi_predictions_slice_250.json",
    500: "/content/Jupiter_Inlet/output_sahi/sahi_predictions_slice_500.json",
    1000: "/content/Jupiter_Inlet/output_sahi/sahi_predictions_slice_1000.json",
}

OUTPUT_FIXED = "/content/Jupiter_Inlet/eval_fixed"
os.makedirs(OUTPUT_FIXED, exist_ok=True)

BOAT_CAT_ID = 1
IOU_THRESHOLD = [0.2, 0.3, 0.5]

#I need to ensure that the boat id is the same for all the annotations
def normalize_coco(input_json, output_json, boat_id=1):
    with open(input_json) as f:
        data = json.load(f)

    if isinstance(data, dict):
        data["categories"] = [{
            "id": boat_id,
            "name": "Boat",
            "supercategory": "Boat"
        }]
        for ann in data.get("annotations", []):
            ann["category_id"] = boat_id
            ann["bbox"] = [float(b) for b in ann["bbox"]]
        out = data

    elif isinstance(data, list):
        for ann in data:
            ann["category_id"] = boat_id
            ann["bbox"] = [float(b) for b in ann["bbox"]]
        out = data

    else:
        raise ValueError("Unsupported COCO format")

    with open(output_json, "w") as f:
        json.dump(out, f, indent=2)

    return output_json

FIXED_GT = normalize_coco(GT_JSON, f"{OUTPUT_FIXED}/gt_fixed.json")

#I am calculating the Intersection over Union (IoU)
def bbox_iou(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[0] + a[2], b[0] + b[2])
    yB = min(a[1] + a[3], b[1] + b[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter / (union + 1e-6)

#Now I am making a confusion matrix for each SAHI window size and IoU threshold
def compute_confusion(coco_gt, coco_pred, iou_thresh):
    y_true, y_pred = [], []

    for img_id in coco_gt.getImgIds():
        gt_ids = coco_gt.getAnnIds(imgIds=[img_id], catIds=[BOAT_CAT_ID])
        pr_ids = coco_pred.getAnnIds(imgIds=[img_id], catIds=[BOAT_CAT_ID])

        gt_boxes = [a["bbox"] for a in coco_gt.loadAnns(gt_ids)]
        pr_boxes = [a["bbox"] for a in coco_pred.loadAnns(pr_ids)]

        matched = set()

        for pb in pr_boxes:
            hit = False
            for i, gb in enumerate(gt_boxes):
                if i in matched:
                    continue
                if bbox_iou(pb, gb) >= iou_thresh:
                    hit = True
                    matched.add(i)
                    break
            y_pred.append(1)
            y_true.append(1 if hit else 0)

        for i in range(len(gt_boxes)):
            if i not in matched:
                y_pred.append(0)
                y_true.append(1)

    return confusion_matrix(y_true, y_pred, labels=[0, 1])

#Checking how many boats were in my ground truth
coco_gt = COCO(FIXED_GT)
gt_count = len(coco_gt.getAnnIds(catIds=[BOAT_CAT_ID]))
print(f"GT boats: {gt_count}")

#Getting the precision, recall, tp, fn and fp for each IoU threshold and sahi window size
for slice_size, path in SAHI_FILES.items():
    fixed_path = normalize_coco(
        path,
        f"{OUTPUT_FIXED}/sahi_{slice_size}_fixed.json"
    )

    with open(fixed_path) as f:
        data = json.load(f)

    coco_sahi = coco_gt.loadRes(
        data if isinstance(data, list) else data["annotations"]
    )

    pred_count = len(coco_sahi.getAnnIds(catIds=[BOAT_CAT_ID]))

    print(f"\n=== SAHI slice {slice_size} ===")
    print(f"Predicted boats: {pred_count}")

    for iou in IOU_THRESHOLDS:
        cm = compute_confusion(coco_gt, coco_sahi, iou)
        tn, fp, fn, tp = cm.ravel()

        print(f"\nIoU ≥ {iou}")
        print(cm)
        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision={tp/(tp+fp+1e-6):.3f}")
        print(f"Recall={tp/(tp+fn+1e-6):.3f}")


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
GT boats: 119
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!

=== SAHI slice 150 ===
Predicted boats: 44

IoU ≥ 0.3
[[  0  35]
 [110   9]]
TP=9, FP=35, FN=110
Precision=0.205
Recall=0.076

IoU ≥ 0.5
[[  0  35]
 [110   9]]
TP=9, FP=35, FN=110
Precision=0.205
Recall=0.076
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!

=== SAHI slice 250 ===
Predicted boats: 68

IoU ≥ 0.3
[[  0  57]
 [108  11]]
TP=11, FP=57, FN=108
Precision=0.162
Recall=0.092

IoU ≥ 0.5
[[  0  58]
 [109  10]]
TP=10, FP=58, FN=109
Precision=0.147
Recall=0.084
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!

=== SAHI slice 500 ===
Predicted boats: 57

IoU ≥ 0.3
[[  0  51]
 [113   6]]
TP=6, FP=51, FN=113
Precision=0.105
Recall=0.050

IoU ≥ 0.5
[[  0  52]
 [114   5]]
TP=5, FP=52, FN=114
Precision=0.088
Recall=0.042
Loading and preparing result